In [16]:
import numpy as np 
import torch 
import torch.nn as nn 
import matplotlib.pyplot as plt 
import gymnasium as gym 
from config import *

In [17]:
import random
import numpy as np
import torch

def set_seed(seed=44):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True)

set_seed(44)

In [ ]:
from dataclasses import dataclass
import numpy as np

@dataclass
class UAV:
    id: int
    position: np.ndarray
    velocity: np.ndarray
    battery_j: float

    active: bool = True

@dataclass
class Target:
    id: int
    position: np.ndarray
    confirmed: bool = False

@dataclass
class Obstacle:
    position: np.ndarray
    radius: float
    height: float

    def __post_init__(self):
        self.position = np.asarray(
            self.position,
            dtype=np.float64,
        )
        self.radius = float(self.radius)
        self.height = float(self.height)

        if self.position.shape != (2,):
            raise ValueError(
                "Obstacle.position must be the XY center with shape (2,)"
            )

        if not np.all(np.isfinite(self.position)):
            raise ValueError(
                "Obstacle.position must contain only finite values"
            )

        if not np.isfinite(self.radius) or self.radius <= 0.0:
            raise ValueError(
                "Obstacle.radius must be finite and > 0"
            )

        if not np.isfinite(self.height) or self.height <= 0.0:
            raise ValueError(
                "Obstacle.height must be finite and > 0"
            )

@dataclass
class Report:
    target_id: int
    source_uav: int
    created_step: int
    size_bytes: int
    ttl_s: float
    delivered_bytes: int = 0

    def __post_init__(self):
        for name, value in (
            ("target_id", self.target_id),
            ("source_uav", self.source_uav),
            ("created_step", self.created_step),
            ("size_bytes", self.size_bytes),
            ("delivered_bytes", self.delivered_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(value, (int, np.integer))
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        self.target_id = int(self.target_id)
        self.source_uav = int(self.source_uav)
        self.created_step = int(self.created_step)
        self.size_bytes = int(self.size_bytes)
        self.delivered_bytes = int(self.delivered_bytes)
        self.ttl_s = float(self.ttl_s)

        if self.target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if self.source_uav < 0:
            raise ValueError(
                "source_uav must be >= 0"
            )

        if self.created_step < 0:
            raise ValueError(
                "created_step must be >= 0"
            )

        if self.size_bytes <= 0:
            raise ValueError(
                "size_bytes must be > 0"
            )

        if (
            not np.isfinite(self.ttl_s)
            or self.ttl_s <= 0.0
        ):
            raise ValueError(
                "ttl_s must be finite and > 0"
            )

        if not (
            0
            <= self.delivered_bytes
            <= self.size_bytes
        ):
            raise ValueError(
                "delivered_bytes must be in "
                "[0, size_bytes]"
            )


In [19]:
def create_uav(rng):
    if not isinstance(rng, np.random.Generator):
        raise TypeError("rng must be numpy.random.Generator")

    num_uavs = int(CONFIG["num_uavs"])
    gcs = np.asarray(CONFIG["gcs_position"], dtype=np.float64)
    launch_radius = float(CONFIG["launch_radius_m"])
    safety_distance = float(CONFIG["safety_distance"])
    launch_min_spacing = float(CONFIG["launch_min_spacing_m"])
    map_size = float(CONFIG["map_size"])
    altitude = float(CONFIG["altitude_min"])

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if gcs.shape != (3,) or not np.all(np.isfinite(gcs)):
        raise ValueError("gcs_position must be a finite 3D position")

    if not (
        0.0 <= gcs[0] <= map_size
        and 0.0 <= gcs[1] <= map_size
    ):
        raise ValueError("gcs_position must lie inside the map")

    if not np.isfinite(launch_radius) or launch_radius <= 0.0:
        raise ValueError("launch_radius_m must be finite and > 0")

    if not np.isfinite(safety_distance) or safety_distance < 0.0:
        raise ValueError("safety_distance must be finite and >= 0")

    if (
        not np.isfinite(launch_min_spacing)
        or launch_min_spacing < safety_distance
    ):
        raise ValueError(
            "launch_min_spacing_m must be finite and >= safety_distance"
        )

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(altitude):
        raise ValueError("altitude_min must be finite")

    uavs = []
    max_attempts = max(10_000, 1_000 * num_uavs)
    attempts = 0
    launch_radius_sq = launch_radius * launch_radius

    while len(uavs) < num_uavs:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all UAVs inside the launch region while "
                "respecting map bounds and launch_min_spacing_m. "
                "Increase launch_radius_m or reduce num_uavs/launch_min_spacing_m."
            )

        offset_xy = rng.uniform(
            -launch_radius,
            launch_radius,
            size=2,
        ).astype(np.float64)

        if float(offset_xy @ offset_xy) > launch_radius_sq:
            continue

        candidate_xy = gcs[:2] + offset_xy

        if not (
            0.0 <= candidate_xy[0] <= map_size
            and 0.0 <= candidate_xy[1] <= map_size
        ):
            continue

        too_close = any(
            np.linalg.norm(candidate_xy - uav.position[:2])
            < launch_min_spacing
            for uav in uavs
        )

        if too_close:
            continue

        position = np.array(
            [candidate_xy[0], candidate_xy[1], altitude],
            dtype=np.float64,
        )

        uavs.append(
            UAV(
                id=len(uavs),
                position=position,
                velocity=np.zeros(3, dtype=np.float64),
                battery_j=CONFIG["battery_j"],
                active=True,
            )
        )

    return uavs


In [ ]:
def point_inside_obstacle(point, obstacles, margin=0.0):
    point = np.asarray(point, dtype=np.float64)
    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError(
            "margin must be finite and >= 0"
        )

    if point.shape == (2,):
        point_xy = point
        z = 0.0
    elif point.shape == (3,):
        point_xy = point[:2]
        z = float(point[2])
    else:
        raise ValueError(
            "point must have shape (2,) or (3,)"
        )

    if not np.all(np.isfinite(point)):
        raise ValueError(
            "point must contain only finite values"
        )

    for obs in obstacles:
        horizontal_distance = np.linalg.norm(
            point_xy - obs.position
        )

        inside_horizontal = (
            horizontal_distance
            <= obs.radius + margin
        )
        inside_vertical = (
            0.0 <= z <= obs.height + margin
        )

        if inside_horizontal and inside_vertical:
            return True

    return False


In [21]:
def create_obstacle(rng, uavs):
    obstacles = []

    map_size = float(CONFIG["map_size"])
    r_min = float(CONFIG["obstacle_radius_min_m"])
    r_max = float(CONFIG["obstacle_radius_max_m"])
    h_min = float(CONFIG["obstacle_height_min_m"])
    h_max = float(CONFIG["obstacle_height_max_m"])

    safety_margin = float(CONFIG["safety_distance"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    gcs_exclusion = float(CONFIG["gcs_exclusion_radius_m"])

    if r_min <= 0.0 or r_max < r_min:
        raise ValueError("obstacle radius range is invalid")
    if 2.0 * r_max > map_size:
        raise ValueError(
            "obstacle_radius_max_m must be <= map_size / 2"
        )

    max_attempts = 100000
    attempts = 0

    while len(obstacles) < CONFIG["num_obstacles"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all obstacles with the current constraints."
            )

        radius = float(rng.uniform(r_min, r_max))
        height = float(rng.uniform(h_min, h_max))

        x = float(rng.uniform(radius, map_size - radius))
        y = float(rng.uniform(radius, map_size - radius))
        center = np.array([x, y], dtype=np.float64)

        distance_to_gcs = np.linalg.norm(center - gcs_xy)
        if distance_to_gcs <= gcs_exclusion + radius:
            continue

        overlaps_launch = any(
            np.linalg.norm(center - uav.position[:2])
            <= radius + safety_margin
            for uav in uavs
        )
        if overlaps_launch:
            continue

        overlaps_obstacle = any(
            np.linalg.norm(center - other.position)
            <= radius + other.radius
            for other in obstacles
        )
        if overlaps_obstacle:
            continue

        obstacles.append(
            Obstacle(
                position=center,
                radius=radius,
                height=height,
            )
        )

    return obstacles


In [22]:
def create_target(rng, obstacles):
    targets = []
    occupied_cells = set()

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    target_exclusion = float(CONFIG["target_exclusion_radius_m"])

    max_attempts = 100000
    attempts = 0

    while len(targets) < CONFIG["num_targets"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all targets with the current constraints."
            )

        position = rng.uniform(0.0,map_size,size=2,).astype(np.float64)

        if point_inside_obstacle(position, obstacles):
            continue

        if np.linalg.norm(position - gcs_xy) <= target_exclusion:
            continue

        gx = int(np.floor(position[0] / cell_size))
        gy = int(np.floor(position[1] / cell_size))
        cell = (gx, gy)

        if cell in occupied_cells:
            continue

        targets.append(
            Target(
                id=len(targets),
                position=position,
                confirmed=False,
            )
        )
        occupied_cells.add(cell)

    return targets


In [23]:
def create_world(seed):
    rng = np.random.default_rng(seed)
    uavs = create_uav(rng)
    obstacles = create_obstacle(rng, uavs)
    targets = create_target(rng, obstacles)

    return rng, uavs, targets, obstacles


In [ ]:
def segment_intersects_obstacle(
    start_position,
    end_position,
    obstacles,
    margin=0.0,
):
    start = np.asarray(
        start_position,
        dtype=np.float64,
    )

    end = np.asarray(
        end_position,
        dtype=np.float64,
    )

    if start.shape != (3,) or end.shape != (3,):
        raise ValueError(
            "start_position and end_position must have shape (3,)"
        )

    if (
        not np.all(np.isfinite(start))
        or not np.all(np.isfinite(end))
    ):
        raise ValueError(
            "start_position and end_position "
            "must contain only finite values"
        )

    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError(
            "margin must be finite and >= 0"
        )

    direction = end - start
    eps = 1e-12

    for obs in obstacles:

        center = np.asarray(obs.position,dtype=np.float64,)

        radius = float(obs.radius) + float(margin)
        height = float(obs.height) + float(margin)

        dz = float(direction[2])

        if abs(dz) <= eps:

            if not (
                0.0 <= start[2] <= height
            ):
                continue

            z_enter = 0.0
            z_exit = 1.0

        else:

            t_ground = (0.0 - start[2]) / dz
            t_top = (height - start[2]) / dz

            z_enter = max(0.0,min(t_ground, t_top))
            z_exit = min(1.0,max(t_ground, t_top))

            if z_enter > z_exit:
                continue

        relative_xy = start[:2] - center[:2]
        direction_xy = direction[:2]

        a = direction_xy@direction_xy
        c = (relative_xy@relative_xy)- radius * radius

        if a <= eps:

            if c > 0:
                continue

            xy_enter = 0
            xy_exit = 1

        else:

            b = 2 * relative_xy@direction_xy
            discriminant = (b * b- 4.0 * a * c)

            if discriminant < 0.0:
                continue

            root = np.sqrt(max(discriminant, 0.0))

            t1 = (-b - root) / (2.0 * a)
            t2 = (-b + root) / (2.0 * a)

            xy_enter = max(0.0,min(t1, t2))
            xy_exit = min(1.0,max(t1, t2))

            if xy_enter > xy_exit:
                continue

        enter = max(z_enter,xy_enter)
        exit_ = min(z_exit,xy_exit)

        if enter <= exit_:
            return True

    return False


In [ ]:
def compute_motion_candidate(
    uav,
    action,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    action = np.asarray(
        action,
        dtype=np.float64,
    )

    if action.shape != (3,):
        raise ValueError("action must have shape (3,)")

    if not np.all(np.isfinite(action)):
        raise ValueError("action must contain only finite values")

    if not uav.active:
        return (
            uav.position.copy(),
            np.zeros(3, dtype=np.float64),
            False,
        )

    action = np.clip(
        action,
        -1.0,
        1.0,
    )

    action_norm = np.linalg.norm(action)

    if action_norm > 1.0:
        action = action / action_norm

    acceleration = action * CONFIG["max_accel"]

    candidate_velocity = (
        uav.velocity
        + acceleration * dt
    )

    speed = np.linalg.norm(candidate_velocity)

    if speed > CONFIG["max_speed"]:
        candidate_velocity = (candidate_velocity/ speed* CONFIG["max_speed"])

    old_position = uav.position.copy()

    raw_candidate_position = old_position+ candidate_velocity * dt

    candidate_position = raw_candidate_position.copy()

    candidate_position[0] = np.clip(
        candidate_position[0],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[1] = np.clip(
        candidate_position[1],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[2] = np.clip(
        candidate_position[2],
        CONFIG["altitude_min"],
        CONFIG["altitude_max"],
    )

    boundary_clipped = not np.allclose(
        raw_candidate_position,
        candidate_position,
    )

    actual_velocity = (
        candidate_position
        - old_position
    ) / dt

    return (
        candidate_position,
        actual_velocity,
        boundary_clipped,
    )


In [26]:
def minimum_distance_during_motion(
    start_a,
    end_a,
    start_b,
    end_b,
):
    start_a = np.asarray(start_a,dtype=np.float64,)
    end_a = np.asarray(end_a,dtype=np.float64,)
    start_b = np.asarray(start_b,dtype=np.float64,)
    end_b = np.asarray(end_b,dtype=np.float64,)
    for point in (
        start_a,
        end_a,
        start_b,
        end_b,
    ):
        if point.shape != (3,):
            raise ValueError(
                "all positions must have shape (3,)"
            )

        if not np.all(np.isfinite(point)):
            raise ValueError(
                "all positions must contain only finite values"
            )
    relative_start = start_a - start_b
    displacement_a = end_a - start_a
    displacement_b = end_b - start_b

    relative_motion = displacement_a - displacement_b

    denominator = relative_motion@relative_motion
    if denominator <= 1e-12:
        return np.linalg.norm(
                relative_start
            )
        

    t_closest = -(relative_start@relative_motion)/ denominator
    t_closest = np.clip(t_closest,0.0,1.0,)

    relative_at_closest = (
        relative_start
        + t_closest * relative_motion
    )

    return float(
        np.linalg.norm(
            relative_at_closest
        )
    )

In [27]:
def apply_swarm_motion(
    uavs,
    actions,
    obstacles=None,
    dt=None,
):
    if obstacles is None:
        obstacles = []

    if dt is None:
        dt = CONFIG["dt"]
    dt = float(dt)


    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    num_uavs = len(uavs)

    if num_uavs == 0:
        raise ValueError(
            "uavs must not be empty"
        )

    actions = np.asarray(actions,dtype=np.float64)

    if actions.shape != (num_uavs, 3):
        raise ValueError(
            f"actions must have shape "
            f"({num_uavs}, 3)"
        )

    if not np.all(np.isfinite(actions)):
        raise ValueError(
            "actions must contain only finite values"
        )

    old_positions = np.stack([
        uav.position.copy()
        for uav in uavs
    ])

    safety_distance = float(CONFIG["safety_distance"])

    for i in range(num_uavs):
        if not uavs[i].active:
            continue

        for j in range(i + 1,num_uavs):
            if not uavs[j].active:
                continue

            distance = np.linalg.norm(old_positions[i]- old_positions[j])

            if distance < safety_distance:
                raise ValueError(
                    "initial active UAV positions "
                    "violate safety_distance"
                )

    candidate_positions = []
    candidate_velocities = []
    boundary_clipped = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, (uav, action) in enumerate(zip(uavs, actions)):
        
        position,velocity,clipped= compute_motion_candidate(
            uav,
            action,
            dt=dt,
        )

        candidate_positions.append(position)
        candidate_velocities.append(velocity)

        boundary_clipped[i] = clipped

    candidate_positions = np.stack(candidate_positions)
    candidate_velocities = np.stack(candidate_velocities)

    blocked_by_obstacle = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, uav in enumerate(uavs):
        if not uav.active:
            continue

        if segment_intersects_obstacle(
            old_positions[i],
            candidate_positions[i],
            obstacles,
        ):
            blocked_by_obstacle[i] = True

    blocked_by_peer = np.zeros(num_uavs,dtype=bool)
    blocked = blocked_by_obstacle.copy()
    

    while True:
        effective_positions = candidate_positions.copy()
        effective_positions[blocked] = old_positions[blocked]

        next_blocked = blocked.copy()

        for i in range(num_uavs):
            if not uavs[i].active:
                continue

            for j in range(i + 1,num_uavs):
                if not uavs[j].active:
                    continue

                distance = minimum_distance_during_motion(
                        old_positions[i],
                        effective_positions[i],
                        old_positions[j],
                        effective_positions[j],
                    )
                
                if distance < safety_distance:
                    next_blocked[i] = True
                    next_blocked[j] = True

                    blocked_by_peer[i] = True
                    blocked_by_peer[j] = True

        if np.array_equal(next_blocked,blocked):
            break

        blocked = next_blocked

    for i, uav in enumerate(uavs):
        if not uav.active:
            uav.velocity = np.zeros(3, dtype=np.float64)
            continue

        if blocked[i]:
            uav.position = old_positions[i].copy()
            uav.velocity = np.zeros(3,dtype=np.float64)

        else:
            uav.position = candidate_positions[i].copy()
            uav.velocity = candidate_velocities[i].copy()

    return {
        "blocked": blocked,
        "blocked_by_obstacle":blocked_by_obstacle,
        "blocked_by_peer":blocked_by_peer,
        "boundary_clipped":boundary_clipped,
    }


In [ ]:
def sensing_profile(altitude):
    altitude = float(altitude)

    if not np.isfinite(altitude):
        raise ValueError("altitude must be finite")

    altitude = np.clip(altitude,CONFIG["altitude_min"],CONFIG["altitude_max"])
    altitude_anchors = np.asarray(CONFIG["altitude_anchors"])

    pd_anchors = np.asarray(CONFIG["pd"])
    pf_anchors = np.asarray(CONFIG["pf"])

    pd = np.interp(altitude,altitude_anchors,pd_anchors)
    pf = np.interp(altitude,altitude_anchors,pf_anchors)

    full_fov = CONFIG["camera_full_fov_deg"]
    half_fov_rad = np.deg2rad(full_fov / 2.0)
    fov_radius = altitude* np.tan(half_fov_rad)

    return (
        float(pd),
        float(pf),
        float(fov_radius)
    )

In [29]:
def create_belief_maps():
    grid_n = int(np.ceil(CONFIG["map_size"] / CONFIG["grid_cell_m"]))
    belief_maps = np.full((CONFIG["num_uavs"],grid_n,grid_n),CONFIG["belief_prior"])
    return belief_maps

In [ ]:
def world_to_grid(position_xy):
    position_xy = np.asarray(position_xy, dtype=np.float64)

    if position_xy.ndim != 1 or position_xy.size < 2:
        raise ValueError(
            "position_xy must be a 1D array containing at least x and y"
        )

    x = float(position_xy[0])
    y = float(position_xy[1])

    if not np.isfinite(x) or not np.isfinite(y):
        raise ValueError("position must be finite")

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(cell_size) or cell_size <= 0.0:
        raise ValueError("grid_cell_m must be finite and > 0")

    if not (
        0.0 <= x <= map_size
        and 0.0 <= y <= map_size
    ):
        raise ValueError("position is outside the map")

    grid_n = int(np.ceil(map_size / cell_size))

    gx = int(np.floor(x / cell_size))
    gy = int(np.floor(y / cell_size))

    gx = min(gx, grid_n - 1)
    gy = min(gy, grid_n - 1)

    return gx, gy


In [31]:
def cell_intersects_fov(
    gx,
    gy,
    uav_xy,
    fov_radius,
):
    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])

    x_min = gx * cell_size
    x_max = min((gx + 1) * cell_size,map_size)

    y_min = gy * cell_size
    y_max = min((gy + 1) * cell_size,map_size)

    closest_x = np.clip(uav_xy[0],x_min,x_max)
    closest_y = np.clip(uav_xy[1],y_min,y_max)

    dx = float(uav_xy[0] - closest_x)
    dy = float(uav_xy[1] - closest_y)

    return (
        dx * dx + dy * dy <= fov_radius * fov_radius
    )

In [32]:
import math


def _circle_sqrt_integral(x, radius):
    """Antiderivative of sqrt(radius^2 - x^2) on [-radius, radius]."""
    x = max(-radius, min(radius, float(x)))
    root = math.sqrt(max(0.0, radius * radius - x * x))
    return 0.5 * (
        x * root
        + radius * radius * math.asin(x / radius)
    )


def _circle_rectangle_intersection_area(
    circle_x,
    circle_y,
    radius,
    x_min,
    x_max,
    y_min,
    y_max,
):
    """Exact area of a circle intersected with an axis-aligned rectangle."""
    radius = float(radius)
    if radius <= 0.0 or x_max <= x_min or y_max <= y_min:
        return 0.0

    # Translate the circle center to the origin.
    x_min = float(x_min) - float(circle_x)
    x_max = float(x_max) - float(circle_x)
    y_min = float(y_min) - float(circle_y)
    y_max = float(y_max) - float(circle_y)

    left = max(x_min, -radius)
    right = min(x_max, radius)

    if right <= left or y_max <= -radius or y_min >= radius:
        return 0.0

    # The integrand changes form only where a horizontal rectangle edge
    # intersects the circle. Split at those x values, then integrate the
    # circle arc analytically on each interval.
    cuts = [left, right]
    for y_edge in (y_min, y_max):
        if abs(y_edge) < radius:
            x_cross = math.sqrt(
                max(0.0, radius * radius - y_edge * y_edge)
            )
            for cut in (-x_cross, x_cross):
                if left < cut < right:
                    cuts.append(cut)

    cuts = sorted(set(cuts))
    area = 0.0

    for a, b in zip(cuts[:-1], cuts[1:]):
        if b <= a:
            continue

        midpoint = 0.5 * (a + b)
        half_height = math.sqrt(
            max(0.0, radius * radius - midpoint * midpoint)
        )

        upper = min(y_max, half_height)
        lower = max(y_min, -half_height)
        if upper <= lower:
            continue

        arc_integral = (
            _circle_sqrt_integral(b, radius)
            - _circle_sqrt_integral(a, radius)
        )

        if y_max < half_height:
            upper_integral = y_max * (b - a)
        else:
            upper_integral = arc_integral

        if y_min > -half_height:
            lower_integral = y_min * (b - a)
        else:
            lower_integral = -arc_integral

        area += upper_integral - lower_integral

    return max(0.0, float(area))


def cell_fov_coverage_fraction(gx, gy, uav_xy, fov_radius):
    """Return the fraction of a belief cell covered by the circular FOV.

    Geometry depends only on UAV/FOV and grid cell; target ground truth never
    changes measurement support. The circle/rectangle area is analytic, which
    is both deterministic and much cheaper than per-cell numerical quadrature.
    """
    if isinstance(gx, (bool, np.bool_)) or not isinstance(gx, (int, np.integer)):
        raise TypeError("gx must be an integer")
    if isinstance(gy, (bool, np.bool_)) or not isinstance(gy, (int, np.integer)):
        raise TypeError("gy must be an integer")

    gx = int(gx)
    gy = int(gy)
    uav_xy = np.asarray(uav_xy, dtype=np.float64)
    fov_radius = float(fov_radius)

    if uav_xy.shape != (2,) or not np.all(np.isfinite(uav_xy)):
        raise ValueError("uav_xy must be a finite shape-(2,) position")
    if not np.isfinite(fov_radius) or fov_radius < 0.0:
        raise ValueError("fov_radius must be finite and >= 0")
    if fov_radius == 0.0:
        return 0.0

    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])
    grid_n = int(np.ceil(map_size / cell_size))

    if not (0 <= gx < grid_n and 0 <= gy < grid_n):
        raise ValueError("grid cell index out of range")

    x_min = gx * cell_size
    x_max = min((gx + 1) * cell_size, map_size)
    y_min = gy * cell_size
    y_max = min((gy + 1) * cell_size, map_size)

    if not cell_intersects_fov(gx, gy, uav_xy, fov_radius):
        return 0.0

    intersection_area = _circle_rectangle_intersection_area(
        circle_x=float(uav_xy[0]),
        circle_y=float(uav_xy[1]),
        radius=fov_radius,
        x_min=x_min,
        x_max=x_max,
        y_min=y_min,
        y_max=y_max,
    )
    cell_area = (x_max - x_min) * (y_max - y_min)

    if cell_area <= 0.0:
        return 0.0

    return float(np.clip(intersection_area / cell_area, 0.0, 1.0))

def cells_with_fov_coverage(uav):
    if not uav.active:
        return []

    altitude = float(uav.position[2])

    if altitude <= 0.0:
        return []

    _, _, fov_radius = sensing_profile(altitude)
    cell_size = float(CONFIG["grid_cell_m"])
    grid_n = int(np.ceil(CONFIG["map_size"] / cell_size))
    uav_xy = np.asarray(uav.position[:2], dtype=np.float64)
    uav_x = float(uav_xy[0])
    uav_y = float(uav_xy[1])

    gx_min = max(0, int(np.floor((uav_x - fov_radius) / cell_size)))
    gx_max = min(grid_n - 1, int(np.floor((uav_x + fov_radius) / cell_size)))
    gy_min = max(0, int(np.floor((uav_y - fov_radius) / cell_size)))
    gy_max = min(grid_n - 1, int(np.floor((uav_y + fov_radius) / cell_size)))

    visible = []
    for gy in range(gy_min, gy_max + 1):
        for gx in range(gx_min, gx_max + 1):
            coverage = cell_fov_coverage_fraction(
                gx,
                gy,
                uav_xy,
                fov_radius,
            )
            if coverage > 0.0:
                visible.append((gx, gy, coverage))

    return visible


def cells_in_fov(uav):
    return [
        (gx, gy)
        for gx, gy, _ in cells_with_fov_coverage(uav)
    ]

In [33]:
def get_target_cells(targets):
    target_cells = {}

    for target in targets:
        gx, gy = world_to_grid(target.position)
        target_cells.setdefault((gx, gy), []).append(target.id)

    return target_cells


In [34]:
def sample_sensor_measurement(has_target,pd,pf,rng):
    if has_target:
        positive_probability = pd

    else:
        positive_probability = pf
    observation = (rng.random()< positive_probability)

    return int(observation)

In [ ]:
def bayes_update(prior, observation, pd, pf, eps=1e-8):
    eps = float(eps)

    if (
        not np.isfinite(eps)
        or not 0.0 < eps < 0.5
    ):
        raise ValueError(
            "eps must be finite and in (0, 0.5)"
        )

    prior = float(prior)
    pd = float(pd)
    pf = float(pf)

    for name, value in (
        ("prior", prior),
        ("pd", pd),
        ("pf", pf),
    ):
        if not np.isfinite(value):
            raise ValueError(
                f"{name} must be finite"
            )

        if not 0.0 <= value <= 1.0:
            raise ValueError(
                f"{name} must be in [0, 1]"
            )

    prior = np.clip(prior,eps,1.0 - eps)

    if observation == 1:
        numerator = pd * prior
        denominator = pd * prior + pf * (1.0 - prior)
    elif observation == 0:
        numerator = (1.0 - pd) * prior
        denominator = (
            (1.0 - pd) * prior
            + (1.0 - pf) * (1.0 - prior)
        )
    else:
        raise ValueError("observation must be 0 or 1")

    denominator = max(float(denominator), eps)
    posterior = numerator / denominator

    return float(np.clip(posterior, eps, 1.0 - eps))


In [36]:
def sense_and_update(uav, belief_map, targets, rng):
    if not uav.active:
        return []

    altitude = float(uav.position[2])

    if altitude <= 0.0:
        return []

    base_pd, base_pf, fov_radius = sensing_profile(altitude)
    visible_cells = cells_with_fov_coverage(uav)
    target_cells = get_target_cells(targets)
    targets_by_id = {target.id: target for target in targets}

    sensing_log = []

    for gx, gy, coverage_fraction in visible_cells:
        candidate_target_ids = target_cells.get((gx, gy), [])
        target_ids_in_fov = [
            target_id
            for target_id in candidate_target_ids
            if np.linalg.norm(
                targets_by_id[target_id].position - uav.position[:2]
            ) <= fov_radius
        ]

        has_target = len(target_ids_in_fov) > 0

        # Scale cell-level evidence by the actually observed cell fraction.
        # At zero coverage the measurement carries no information; at full
        # coverage this reduces exactly to the original Pd/Pf model.
        effective_pf = float(
            1.0 - (1.0 - base_pf) ** coverage_fraction
        )
        effective_pd = float(
            coverage_fraction * base_pd
            + (1.0 - coverage_fraction) * effective_pf
        )

        if has_target:
            # The simulator knows the target's true continuous position is
            # inside the footprint, so the physical detector uses base Pd.
            positive_probability = base_pd
        else:
            # False alarms scale with the observed fraction of the cell.
            positive_probability = effective_pf

        observation = int(rng.random() < positive_probability)

        prior = float(belief_map[gy, gx])
        posterior = bayes_update(
            prior,
            observation,
            effective_pd,
            effective_pf,
        )
        belief_map[gy, gx] = posterior

        sensing_log.append({
            "gx": gx,
            "gy": gy,
            "target_ids": target_ids_in_fov,
            "has_target": has_target,
            "observation": observation,
            "prior": prior,
            "posterior": posterior,
            "pd": effective_pd,
            "pf": effective_pf,
            "base_pd": base_pd,
            "base_pf": base_pf,
            "coverage_fraction": float(coverage_fraction),
        })

    return sensing_log

In [37]:
def check_confirmation(uav, sensing_record):
    if not uav.active:
        return "none", []

    posterior = float(sensing_record["posterior"])
    observation = int(sensing_record["observation"])
    target_ids = list(sensing_record["target_ids"])

    if posterior < CONFIG["confirmation_threshold"]:
        return "none", []

    if observation != 1:
        return "none", []

    if target_ids:
        return "true_confirmation", target_ids

    return "false_confirmation", []

In [38]:
@dataclass
class ConfirmationEvent:
    target_id: int | None
    uav_id: int
    step: int

    cell: tuple
    uav_position: np.ndarray

    altitude: float
    belief: float
    observation: int

    confirmation_type: str

In [39]:
def create_confirmation_events(
    uav,
    sensing_record,
    step
):
    confirmation_type, target_ids = check_confirmation(
        uav,
        sensing_record
    )

    if confirmation_type == "none":
        return []

    common = dict(
        uav_id=int(uav.id),
        step=int(step),

        cell=(
            int(sensing_record["gx"]),
            int(sensing_record["gy"]),
        ),

        uav_position=uav.position.copy(),

        altitude=float(uav.position[2]),
        belief=float(sensing_record["posterior"]),
        observation=int(sensing_record["observation"]),
        confirmation_type=confirmation_type,
    )

    if confirmation_type == "false_confirmation":
        return [ConfirmationEvent(target_id=None,**common,)]

    return [ConfirmationEvent(target_id=int(target_id),**common,) for target_id in target_ids]

In [40]:
def process_confirmation_events(
    events,
    targets,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
):
    reports = []
    event_log = []

    if not isinstance(report_buffers, list):
        raise TypeError("report_buffers must be a list")
    if not isinstance(pending_reports, list):
        raise TypeError("pending_reports must be a list")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    targets_by_id = {
        target.id: target
        for target in targets
    }

    generated_target_ids = set()

    for event in events:
        event_log.append(event)

        if event.confirmation_type != "true_confirmation":
            continue

        if event.target_id not in targets_by_id:
            raise ValueError("confirmation event target_id not found")

        target = targets_by_id[event.target_id]
        target.confirmed = True

        # Expired network state must not suppress a fresh observation.
        # Clean stale buffered/pending copies using the observation step
        # before duplicate checks, so correctness does not depend on caller
        # cleanup order.
        current_step = int(event.step)
        for buffer in report_buffers:
            remove_expired_reports(buffer, current_step)
        remove_expired_reports(pending_reports, current_step)

        if target.id in gcs_received_target_ids:
            continue

        report_already_buffered = any(
            report.target_id == target.id
            for buffer in report_buffers
            for report in buffer
        )
        report_already_pending = any(
            report.target_id == target.id
            for report in pending_reports
        )

        if (
            report_already_buffered
            or report_already_pending
            or target.id in generated_target_ids
        ):
            continue

        source_uav = int(event.uav_id)
        if not 0 <= source_uav < len(report_buffers):
            raise ValueError("confirmation event uav_id out of range")

        report = Report(
            target_id=target.id,
            source_uav=source_uav,
            created_step=event.step,
            size_bytes=CONFIG["report_bytes"],
            ttl_s=CONFIG["report_ttl"],
        )

        enqueued, reason = enqueue_report(
            report_buffers[source_uav],
            report,
        )

        if not enqueued:
            if reason == "buffer_full":
                pending_reports.append(report)
            elif reason != "duplicate":
                raise RuntimeError(f"unexpected enqueue result: {reason}")

        reports.append(report)
        generated_target_ids.add(target.id)

    return reports, event_log

In [41]:
def create_report_buffers():
    return [[] for _ in range(CONFIG["num_uavs"])]


def create_pending_reports():
    return []


def create_gcs_received_target_ids():
    return set()

In [42]:
def report_remaining_bytes(report):
    remaining = report.size_bytes - report.delivered_bytes
    return max(0, int(remaining))


def report_is_complete(report):
    return report_remaining_bytes(report) == 0


def buffer_used_bytes(buffer):
    return sum(report.size_bytes for report in buffer)

In [43]:
def report_exists(buffer, target_id):
    return any(
        report.target_id == target_id for report in buffer
    )


def report_exists_in_buffers(report_buffers, target_id):
    return any(
        report_exists(buffer, target_id)
        for buffer in report_buffers
    )


def pending_report_exists(pending_reports, target_id):
    return any(
        report.target_id == target_id
        for report in pending_reports
    )

In [44]:
def enqueue_report(buffer, report):
    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")
    if not isinstance(report, Report):
        raise TypeError("report must be a Report")

    # A report whose bytes have all reached the GCS is retained until
    # mark_target_delivered_to_gcs() records mission-level delivery and
    # removes the report. This prevents losing delivery state between steps.
    if report_exists(buffer, report.target_id):
        return False, "duplicate"

    used_bytes = buffer_used_bytes(buffer)
    new_used_bytes = used_bytes + report.size_bytes

    if new_used_bytes > CONFIG["buffer_bytes"]:
        return False, "buffer_full"

    buffer.append(report)
    return True, "enqueued"

In [ ]:
def report_age_s(report, current_step):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    created_step = report.created_step

    if (
        isinstance(created_step, (bool, np.bool_))
        or not isinstance(created_step, (int, np.integer))
    ):
        raise TypeError("report.created_step must be an integer")

    current_step = int(current_step)
    created_step = int(created_step)

    if created_step < 0:
        raise ValueError("report.created_step must be >= 0")

    if current_step < created_step:
        raise ValueError(
            "current_step must be >= report.created_step"
        )

    dt = float(CONFIG["dt"])

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError("CONFIG['dt'] must be finite and > 0")

    return float(
        (current_step - created_step) * dt
    )


In [46]:
def report_is_expired(report, current_step):
    age_s = report_age_s(report,current_step)

    return age_s >= report.ttl_s

In [47]:
def remove_expired_reports(
    buffer,
    current_step,
):
    kept_reports = []
    expired_reports = []

    for report in buffer:
        if report_is_expired(report, current_step):
            expired_reports.append(report)
        else:
            kept_reports.append(report)

    buffer[:] = kept_reports
    return expired_reports


def remove_completed_reports(buffer):
    completed_reports = [
        report
        for report in buffer
        if report_is_complete(report)
    ]
    buffer[:] = [
        report
        for report in buffer
        if not report_is_complete(report)
    ]
    return completed_reports


def cleanup_report_buffer(buffer, current_step):
    expired = remove_expired_reports(buffer, current_step)
    completed = [
        report
        for report in buffer
        if report_is_complete(report)
    ]
    # Completed reports are intentionally retained until the GCS-delivery
    # state is marked. FIFO selection skips them without deleting them.
    return {
        "expired": expired,
        "completed": completed,
    }


def flush_pending_reports(
    pending_reports,
    report_buffers,
    current_step,
    gcs_received_target_ids,
    uavs,
):
    if not isinstance(pending_reports, list):
        raise TypeError("pending_reports must be a list")
    if not isinstance(report_buffers, list):
        raise TypeError("report_buffers must be a list")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")
    if not isinstance(uavs, list):
        raise TypeError("uavs must be a list")

    uavs_by_id = {}
    for uav in uavs:
        if not isinstance(uav, UAV):
            raise TypeError("uavs must contain only UAV objects")
        uav_id = int(uav.id)
        if uav_id in uavs_by_id:
            raise ValueError("UAV ids must be unique")
        uavs_by_id[uav_id] = uav

    for buffer in report_buffers:
        cleanup_report_buffer(buffer, current_step)

    kept_pending = []
    enqueued_target_ids = []
    expired_target_ids = []
    discarded_target_ids = []
    deferred_inactive_target_ids = []

    for report in pending_reports:
        if report.target_id in gcs_received_target_ids:
            discarded_target_ids.append(report.target_id)
            continue

        if report_is_expired(report, current_step):
            expired_target_ids.append(report.target_id)
            continue

        if report_exists_in_buffers(report_buffers, report.target_id):
            discarded_target_ids.append(report.target_id)
            continue

        source_uav = int(report.source_uav)
        if not 0 <= source_uav < len(report_buffers):
            raise ValueError("pending report source_uav out of range")
        if source_uav not in uavs_by_id:
            raise ValueError("pending report source_uav not found")

        if not uavs_by_id[source_uav].active:
            kept_pending.append(report)
            deferred_inactive_target_ids.append(report.target_id)
            continue

        enqueued, reason = enqueue_report(
            report_buffers[source_uav],
            report,
        )

        if enqueued:
            enqueued_target_ids.append(report.target_id)
        elif reason == "buffer_full":
            kept_pending.append(report)
        elif reason == "duplicate":
            discarded_target_ids.append(report.target_id)
        else:
            raise RuntimeError(f"unexpected enqueue result: {reason}")

    pending_reports[:] = kept_pending

    return {
        "enqueued_target_ids": enqueued_target_ids,
        "expired_target_ids": expired_target_ids,
        "discarded_target_ids": discarded_target_ids,
        "deferred_inactive_target_ids": deferred_inactive_target_ids,
    }

def mark_target_delivered_to_gcs(
    target_id,
    gcs_received_target_ids,
    report_buffers,
    pending_reports,
):
    if isinstance(target_id, (bool, np.bool_)) or not isinstance(
        target_id,
        (int, np.integer),
    ):
        raise TypeError("target_id must be an integer")

    target_id = int(target_id)
    if target_id < 0:
        raise ValueError("target_id must be >= 0")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    gcs_received_target_ids.add(target_id)

    removed_from_buffers = 0
    for buffer in report_buffers:
        before = len(buffer)
        buffer[:] = [
            report
            for report in buffer
            if report.target_id != target_id
        ]
        removed_from_buffers += before - len(buffer)

    before_pending = len(pending_reports)
    pending_reports[:] = [
        report
        for report in pending_reports
        if report.target_id != target_id
    ]

    return {
        "removed_from_buffers": removed_from_buffers,
        "removed_from_pending": before_pending - len(pending_reports),
    }

In [48]:
def distance_3d(position_a, position_b):
    position_a = np.asarray(position_a,dtype=np.float64)
    position_b = np.asarray(position_b, dtype=np.float64)

    if position_a.shape != (3,):
        raise ValueError("position_a must have shape (3,)")

    if position_b.shape != (3,):
        raise ValueError("position_b must have shape (3,)")

    if not np.all(np.isfinite(position_a)):
        raise ValueError("position_a must contain only finite values")

    if not np.all(np.isfinite(position_b)):
        raise ValueError("position_b must contain only finite values")

    return np.linalg.norm(position_a - position_b)
    

In [49]:
def positions_can_communicate(position_a,position_b,max_range_m):
    max_range_m = float(max_range_m)

    if (
        not np.isfinite(max_range_m)
        or max_range_m <= 0.0
    ):
        raise ValueError(
            "max_range_m must be finite and > 0"
        )

    return (distance_3d(position_a,position_b)<= max_range_m)

In [50]:
def uavs_can_communicate(uav_a,uav_b):
    if uav_a.id == uav_b.id:
        return False

    if not uav_a.active:
        return False

    if not uav_b.active:
        return False

    return positions_can_communicate(
        uav_a.position,
        uav_b.position,
        CONFIG["peer_contact_range_m"]
    )

In [51]:
def uav_can_reach_gcs(uav):
    if not uav.active:
        return False

    return positions_can_communicate(
        uav.position,
        CONFIG["gcs_position"],
        CONFIG["gcs_contact_range_m"],
    )

In [52]:
def get_uav_neighbors(uav,uavs):
    neighbors = []

    for other in uavs:
        if uavs_can_communicate(uav,other):
            neighbors.append(other.id)

    return neighbors

In [53]:
GCS_NODE = -1
SILENT_DESTINATION = None

In [54]:
def communication_choices(sender_id,num_uavs=None):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (isinstance(sender_id, bool) or not isinstance(sender_id,(int, np.integer))):
        raise TypeError("sender_id must be an integer")

    if (isinstance(num_uavs, bool) or not isinstance(num_uavs,(int, np.integer))):
        raise TypeError("num_uavs must be an integer")

    sender_id = int(sender_id)
    num_uavs = int(num_uavs)

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if not 0 <= sender_id < num_uavs:
        raise ValueError("sender_id out of range")

    peers = tuple(
        uav_id
        for uav_id in range(num_uavs)
        if uav_id != sender_id
    )

    return (SILENT_DESTINATION,*peers,GCS_NODE)

In [ ]:
def decode_destination(
    sender_id,
    destination_index,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (
        isinstance(
            destination_index,
            (bool, np.bool_),
        )
        or not isinstance(
            destination_index,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "destination_index must be an integer"
        )

    destination_index = int(destination_index)

    choices = communication_choices(
        sender_id,
        num_uavs,
    )

    if not 0<= destination_index< len(choices):
        raise ValueError(
            "destination_index out of range"
        )

    return choices[destination_index]

In [ ]:
def decode_tx_power_w(power_action):
    power_array = np.asarray(power_action)

    if power_array.shape not in [(), (1,)]:
        raise ValueError(
            "power_action must be a scalar or have shape (1,)"
        )

    if (
        np.issubdtype(power_array.dtype, np.bool_)
        or not np.issubdtype(power_array.dtype, np.number)
    ):
        raise TypeError(
            "power_action must be numeric and not boolean"
        )

    power_value = float(power_array.item())

    if not np.isfinite(power_value):
        raise ValueError(
            "power_action must be finite"
        )

    if not -1.0 <= power_value <= 1.0:
        raise ValueError(
            "power_action must be within [-1, 1]"
        )

    power_min = float(CONFIG["tx_power_min_w"])
    power_max = float(CONFIG["tx_power_max_w"])

    if (
        not np.isfinite(power_min)
        or not np.isfinite(power_max)
        or power_min <= 0.0
        or power_max < power_min
    ):
        raise ValueError(
            "invalid TX power range"
        )

    normalized = (power_value + 1.0) / 2.0

    return float(
        power_min
        + normalized
        * (power_max - power_min)
    )


In [60]:
@dataclass
class HybridAction:
    movement: np.ndarray
    destination: int | None
    tx_power_w: float

In [ ]:
def decode_hybrid_action(
    sender_id,
    action,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if not isinstance(action, dict):
        raise TypeError(
            "action must be a dict"
        )

    required_keys = {
        "motion",
        "destination",
        "power",
    }

    if set(action.keys()) != required_keys:
        raise ValueError(
            "action must contain exactly "
            "motion, destination, power"
        )

    motion_object = np.asarray(
        action["motion"],
        dtype=object,
    )

    if motion_object.shape != (3,):
        raise ValueError(
            "motion must have shape (3,)"
        )

    for value in motion_object:
        if isinstance(value, (bool, np.bool_)):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

        if not isinstance(
            value,
            (int, float, np.integer, np.floating),
        ):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

    motion = motion_object.astype(
        np.float64,
    )

    if not np.all(np.isfinite(motion)):
        raise ValueError(
            "motion must contain only finite values"
        )

    if (
        np.any(motion < -1.0)
        or np.any(motion > 1.0)
    ):
        raise ValueError(
            "motion must be within [-1, 1]"
        )

    destination = decode_destination(
        sender_id,
        action["destination"],
        num_uavs,
    )

    decoded_power_w = decode_tx_power_w(action["power"])

    if destination is SILENT_DESTINATION:
        tx_power_w = 0.0
    else:
        tx_power_w = decoded_power_w

    return HybridAction(
        movement=motion.copy(),
        destination=destination,
        tx_power_w=tx_power_w,
    )


In [ ]:
def select_report_for_transmission(
    buffer,
    current_step,
):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    current_step = int(current_step)

    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")

    for report in buffer:
        if not isinstance(report, Report):
            raise TypeError("buffer must contain only Report objects")

    cleanup_report_buffer(buffer, current_step)

    for report in buffer:
        if not report_is_complete(report):
            return report

    return None

In [ ]:
@dataclass(frozen=True)
class TransmissionIntent:
    sender: int
    recipient: int
    target_id: int
    requested_bytes: int
    tx_power_w: float

    def __post_init__(self):
        for name, value in (
            ("sender", self.sender),
            ("recipient", self.recipient),
            ("target_id", self.target_id),
            ("requested_bytes", self.requested_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(
                    value,
                    (int, np.integer),
                )
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        sender = int(self.sender)
        recipient = int(self.recipient)
        target_id = int(self.target_id)
        requested_bytes = int(
            self.requested_bytes
        )
        tx_power_w = float(
            self.tx_power_w
        )

        num_uavs = int(
            CONFIG["num_uavs"]
        )

        if not 0 <= sender < num_uavs:
            raise ValueError(
                "sender out of range"
            )

        if (
            recipient != GCS_NODE
            and not 0 <= recipient < num_uavs
        ):
            raise ValueError(
                "recipient must be GCS_NODE "
                "or a valid UAV id"
            )

        if recipient == sender:
            raise ValueError(
                "sender and recipient "
                "must be different"
            )

        if target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if requested_bytes <= 0:
            raise ValueError(
                "requested_bytes must be > 0"
            )

        power_min = float(
            CONFIG["tx_power_min_w"]
        )
        power_max = float(
            CONFIG["tx_power_max_w"]
        )

        if (
            not np.isfinite(tx_power_w)
            or not power_min
            <= tx_power_w
            <= power_max
        ):
            raise ValueError(
                "tx_power_w outside "
                "configured range"
            )

        object.__setattr__(
            self,
            "sender",
            sender,
        )
        object.__setattr__(
            self,
            "recipient",
            recipient,
        )
        object.__setattr__(
            self,
            "target_id",
            target_id,
        )
        object.__setattr__(
            self,
            "requested_bytes",
            requested_bytes,
        )
        object.__setattr__(
            self,
            "tx_power_w",
            tx_power_w,
        )

In [ ]:
def build_transmission_intent(
    sender_id,
    hybrid_action,
    buffer,
    current_step,
    peer_transfer_states=None,
):
    if (
        isinstance(
            sender_id,
            (bool, np.bool_),
        )
        or not isinstance(
            sender_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "sender_id must be an integer"
        )

    sender_id = int(sender_id)

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if not 0 <= sender_id < num_uavs:
        raise ValueError(
            "sender_id out of range"
        )

    if not isinstance(
        hybrid_action,
        HybridAction,
    ):
        raise TypeError(
            "hybrid_action must be HybridAction"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if not isinstance(
        buffer,
        list,
    ):
        raise TypeError(
            "buffer must be a list"
        )

    for report in buffer:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "buffer must contain only "
                "Report objects"
            )

    if (
        peer_transfer_states is not None
        and not isinstance(
            peer_transfer_states,
            dict,
        )
    ):
        raise TypeError(
            "peer_transfer_states must be "
            "a dict or None"
        )

    if (
        hybrid_action.destination
        is SILENT_DESTINATION
    ):
        return None

    cleanup_report_buffer(
        buffer,
        current_step,
    )

    destination = (
        hybrid_action.destination
    )

    if destination == GCS_NODE:
        for report in buffer:
            if report_is_complete(
                report
            ):
                continue

            requested_bytes = (
                report_remaining_bytes(
                    report
                )
            )

            if requested_bytes <= 0:
                continue

            return TransmissionIntent(
                sender=sender_id,
                recipient=destination,
                target_id=report.target_id,
                requested_bytes=requested_bytes,
                tx_power_w=(
                    hybrid_action.tx_power_w
                ),
            )

        return None

    # Peer destination: preserve FIFO order, but skip any report
    # that this exact peer already received completely.
    for report in buffer:
        if report_is_complete(
            report
        ):
            continue

        requested_bytes = int(
            report.size_bytes
        )

        if peer_transfer_states is not None:
            key = peer_transfer_key(
                sender_id,
                destination,
                report.target_id,
            )

            state = (
                peer_transfer_states.get(
                    key
                )
            )

            if state is not None:
                if not isinstance(
                    state,
                    PeerTransferState,
                ):
                    raise TypeError(
                        "peer_transfer_states "
                        "values must be "
                        "PeerTransferState"
                    )

                if peer_transfer_is_expired(
                    state,
                    current_step,
                ):
                    del peer_transfer_states[
                        key
                    ]
                    state = None

            if state is not None:
                if (
                    state.source_uav
                    != report.source_uav
                    or state.created_step
                    != report.created_step
                    or state.size_bytes
                    != report.size_bytes
                    or not np.isclose(
                        state.ttl_s,
                        report.ttl_s,
                    )
                ):
                    raise ValueError(
                        "peer transfer state "
                        "does not match "
                        "sender report"
                    )

                requested_bytes = (
                    peer_transfer_remaining_bytes(
                        state
                    )
                )

                if requested_bytes <= 0:
                    # This peer already has the full report.
                    # Continue FIFO search so later reports are
                    # not head-of-line blocked.
                    continue

        return TransmissionIntent(
            sender=sender_id,
            recipient=destination,
            target_id=report.target_id,
            requested_bytes=(
                requested_bytes
            ),
            tx_power_w=(
                hybrid_action.tx_power_w
            ),
        )

    return None


In [ ]:
def transmission_is_feasible(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if len(uavs) != num_uavs:
        raise ValueError(
            "uavs length does not match "
            "CONFIG['num_uavs']"
        )

    for uav in uavs:
        if not isinstance(uav, UAV):
            raise TypeError(
                "uavs must contain only UAV objects"
            )

        if (
            isinstance(uav.id, (bool, np.bool_))
            or not isinstance(
                uav.id,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "each UAV id must be an integer"
            )

    ids = [
        int(uav.id)
        for uav in uavs
    ]

    if len(set(ids)) != num_uavs:
        raise ValueError(
            "UAV ids must be unique"
        )

    if set(ids) != set(range(num_uavs)):
        raise ValueError(
            "UAV ids must be exactly "
            "0..num_uavs-1"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
    }

    sender = uavs_by_id[
        intent.sender
    ]

    if not sender.active:
        return False

    if intent.recipient == GCS_NODE:
        return bool(
            uav_can_reach_gcs(
                sender
            )
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return bool(
        uavs_can_communicate(
            sender,
            recipient,
        )
    )


In [ ]:
LOS_LINK = "los"
NLOS_LINK = "nlos"


def positions_have_los(
    position_a,
    position_b,
    obstacles,
):
    if obstacles is None:
        obstacles = []

    if not isinstance(obstacles, list):
        raise TypeError("obstacles must be a list")

    for obstacle in obstacles:
        if not isinstance(obstacle, Obstacle):
            raise TypeError(
                "obstacles must contain only Obstacle objects"
            )

    return not segment_intersects_obstacle(
        position_a,
        position_b,
        obstacles,
    )


def classify_link_state(
    position_a,
    position_b,
    obstacles,
):
    if positions_have_los(
        position_a,
        position_b,
        obstacles,
    ):
        return LOS_LINK

    return NLOS_LINK


In [ ]:
def db_to_linear(db_value):
    db_value = float(db_value)

    if not np.isfinite(db_value):
        raise ValueError(
            "db_value must be finite"
        )

    return float(
        10.0 ** (db_value / 10.0)
    )


def dbm_to_w(dbm_value):
    dbm_value = float(dbm_value)

    if not np.isfinite(dbm_value):
        raise ValueError(
            "dbm_value must be finite"
        )

    return float(
        10.0 ** (
            (dbm_value - 30.0) / 10.0
        )
    )


def calculate_link_snr(
    tx_power_w,
    distance_m,
    additional_loss_db=0.0,
):
    tx_power_w = float(tx_power_w)
    distance_m = float(distance_m)
    additional_loss_db = float(additional_loss_db)

    if (
        not np.isfinite(tx_power_w)
        or tx_power_w <= 0.0
    ):
        raise ValueError(
            "tx_power_w must be finite and > 0"
        )

    if (
        not np.isfinite(distance_m)
        or distance_m < 0.0
    ):
        raise ValueError(
            "distance_m must be finite and >= 0"
        )

    if (
        not np.isfinite(additional_loss_db)
        or additional_loss_db < 0.0
    ):
        raise ValueError(
            "additional_loss_db must be finite and >= 0"
        )

    power_min = float(
        CONFIG["tx_power_min_w"]
    )
    power_max = float(
        CONFIG["tx_power_max_w"]
    )

    if not (
        power_min
        <= tx_power_w
        <= power_max
    ):
        raise ValueError(
            "tx_power_w outside configured range"
        )

    reference_distance_m = float(
        CONFIG["comm_reference_distance_m"]
    )
    path_loss_exponent = float(
        CONFIG["comm_path_loss_exponent"]
    )

    if (
        not np.isfinite(reference_distance_m)
        or reference_distance_m <= 0.0
    ):
        raise ValueError(
            "comm_reference_distance_m "
            "must be finite and > 0"
        )

    if (
        not np.isfinite(path_loss_exponent)
        or path_loss_exponent <= 0.0
    ):
        raise ValueError(
            "comm_path_loss_exponent "
            "must be finite and > 0"
        )

    reference_gain_linear = db_to_linear(
        CONFIG["comm_reference_gain_db"]
    )

    noise_power_w = dbm_to_w(
        CONFIG["comm_noise_power_dbm"]
    )

    if (
        reference_gain_linear <= 0.0
        or noise_power_w <= 0.0
    ):
        raise ValueError(
            "invalid communication model"
        )

    effective_distance_m = max(
        distance_m,
        reference_distance_m,
    )

    channel_power_gain = (
        reference_gain_linear
        * (
            reference_distance_m
            / effective_distance_m
        )
        ** path_loss_exponent
    )

    additional_gain = db_to_linear(
        -additional_loss_db
    )

    received_power_w = (
        tx_power_w
        * channel_power_gain
        * additional_gain
    )

    snr_linear = (
        received_power_w
        / noise_power_w
    )

    return float(snr_linear)


def snr_linear_to_db(snr_linear):
    snr_linear = float(snr_linear)

    if not np.isfinite(snr_linear) or snr_linear < 0.0:
        raise ValueError(
            "snr_linear must be finite and >= 0"
        )

    if snr_linear == 0.0:
        return float("-inf")

    return float(
        10.0 * np.log10(snr_linear)
    )


In [ ]:
def transmission_distance_m(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    if intent.sender not in uavs_by_id:
        raise ValueError(
            "sender UAV not found"
        )

    sender = uavs_by_id[
        intent.sender
    ]

    if intent.recipient == GCS_NODE:
        return distance_3d(
            sender.position,
            CONFIG["gcs_position"],
        )

    if intent.recipient not in uavs_by_id:
        raise ValueError(
            "recipient UAV not found"
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return distance_3d(
        sender.position,
        recipient.position,
    )


def calculate_intent_snr(
    intent,
    uavs,
    obstacles=None,
):
    if obstacles is None:
        obstacles = []

    if not transmission_is_feasible(
        intent,
        uavs,
    ):
        return 0.0

    distance_m = transmission_distance_m(
        intent,
        uavs,
    )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    sender_position = uavs_by_id[
        intent.sender
    ].position

    if intent.recipient == GCS_NODE:
        recipient_position = np.asarray(
            CONFIG["gcs_position"],
            dtype=np.float64,
        )
    else:
        recipient_position = uavs_by_id[
            intent.recipient
        ].position

    link_state = classify_link_state(
        sender_position,
        recipient_position,
        obstacles,
    )

    if link_state == LOS_LINK:
        additional_loss_db = 0.0
    else:
        additional_loss_db = float(
            CONFIG["comm_nlos_additional_loss_db"]
        )

    return calculate_link_snr(
        intent.tx_power_w,
        distance_m,
        additional_loss_db=additional_loss_db,
    )


In [ ]:
def calculate_link_rate_bps(snr_linear):
    snr_linear = float(snr_linear)

    if not np.isfinite(snr_linear) or snr_linear < 0.0:
        raise ValueError(
            "snr_linear must be finite and >= 0"
        )

    bandwidth_hz = float(
        CONFIG["comm_bandwidth_hz"]
    )

    if (
        not np.isfinite(bandwidth_hz)
        or bandwidth_hz <= 0.0
    ):
        raise ValueError(
            "comm_bandwidth_hz must be finite and > 0"
        )

    if snr_linear == 0.0:
        return 0.0

    return float(
        bandwidth_hz
        * np.log1p(snr_linear)
        / np.log(2.0)
    )


def calculate_intent_rate_bps(
    intent,
    uavs,
    obstacles=None,
):
    snr_linear = calculate_intent_snr(
        intent,
        uavs,
        obstacles=obstacles,
    )

    return calculate_link_rate_bps(
        snr_linear
    )


def calculate_intent_tx_bytes(
    intent,
    uavs,
    obstacles=None,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    rate_bps = calculate_intent_rate_bps(
        intent,
        uavs,
        obstacles=obstacles,
    )

    capacity_bytes = (
        rate_bps
        * dt
        / 8.0
    )

    transferable_bytes = int(
        np.floor(capacity_bytes)
    )

    return min(
        int(intent.requested_bytes),
        transferable_bytes,
    )


In [56]:
@dataclass
class PeerTransferState:
    sender: int
    recipient: int
    target_id: int
    source_uav: int
    created_step: int
    size_bytes: int
    ttl_s: float
    received_bytes: int = 0

    def __post_init__(self):
        for name, value in (
            ("sender", self.sender),
            ("recipient", self.recipient),
            ("target_id", self.target_id),
            ("source_uav", self.source_uav),
            ("created_step", self.created_step),
            ("size_bytes", self.size_bytes),
            ("received_bytes", self.received_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(value, (int, np.integer))
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        self.sender = int(self.sender)
        self.recipient = int(self.recipient)
        self.target_id = int(self.target_id)
        self.source_uav = int(self.source_uav)
        self.created_step = int(self.created_step)
        self.size_bytes = int(self.size_bytes)
        self.received_bytes = int(self.received_bytes)
        self.ttl_s = float(self.ttl_s)

        num_uavs = int(CONFIG["num_uavs"])

        if not 0 <= self.sender < num_uavs:
            raise ValueError("sender out of range")

        if not 0 <= self.recipient < num_uavs:
            raise ValueError("recipient out of range")

        if self.sender == self.recipient:
            raise ValueError(
                "sender and recipient must be different"
            )

        if self.target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if not 0 <= self.source_uav < num_uavs:
            raise ValueError(
                "source_uav out of range"
            )

        if self.created_step < 0:
            raise ValueError(
                "created_step must be >= 0"
            )

        if self.size_bytes <= 0:
            raise ValueError(
                "size_bytes must be > 0"
            )

        if (
            not np.isfinite(self.ttl_s)
            or self.ttl_s <= 0.0
        ):
            raise ValueError(
                "ttl_s must be finite and > 0"
            )

        if not (
            0
            <= self.received_bytes
            <= self.size_bytes
        ):
            raise ValueError(
                "received_bytes must be in "
                "[0, size_bytes]"
            )



In [ ]:
def create_peer_transfer_states():
    return {}


def peer_transfer_key(
    sender,
    recipient,
    target_id,
):
    for name, value in (
        ("sender", sender),
        ("recipient", recipient),
        ("target_id", target_id),
    ):
        if (
            isinstance(value, (bool, np.bool_))
            or not isinstance(value, (int, np.integer))
        ):
            raise TypeError(
                f"{name} must be an integer"
            )

    sender = int(sender)
    recipient = int(recipient)
    target_id = int(target_id)

    num_uavs = int(CONFIG["num_uavs"])

    if not 0 <= sender < num_uavs:
        raise ValueError(
            "sender out of range"
        )

    if not 0 <= recipient < num_uavs:
        raise ValueError(
            "recipient out of range"
        )

    if sender == recipient:
        raise ValueError(
            "sender and recipient must be different"
        )

    if target_id < 0:
        raise ValueError(
            "target_id must be >= 0"
        )

    return (
        sender,
        recipient,
        target_id,
    )

In [ ]:
def peer_transfer_remaining_bytes(state):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    return max(
        0,
        int(
            state.size_bytes
            - state.received_bytes
        ),
    )


def peer_transfer_is_complete(state):
    return (peer_transfer_remaining_bytes(state)== 0)


def peer_transfer_age_s(
    state,
    current_step,
):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < state.created_step:
        raise ValueError(
            "current_step must be >= "
            "state.created_step"
        )

    dt = float(CONFIG["dt"])

    if (
        not np.isfinite(dt)
        or dt <= 0.0
    ):
        raise ValueError(
            "CONFIG['dt'] must be finite and > 0"
        )

    return float((current_step - state.created_step)* dt)

In [ ]:
def peer_transfer_is_expired(
    state,
    current_step,
):
    return peer_transfer_age_s(state,current_step)>= state.ttl_s


def find_report_in_buffer(
    buffer,
    target_id,
):
    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")

    if (
        isinstance(target_id, (bool, np.bool_))
        or not isinstance(
            target_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "target_id must be an integer"
        )

    target_id = int(target_id)

    for report in buffer:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "buffer must contain only "
                "Report objects"
            )

        if report.target_id == target_id:
            return report

    return None





In [ ]:
def get_or_create_peer_transfer_state(
    intent,
    sender_buffer,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if intent.recipient == GCS_NODE:
        raise ValueError(
            "peer transfer state is only "
            "for UAV-to-UAV transmissions"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    report = find_report_in_buffer(
        sender_buffer,
        intent.target_id,
    )

    if report is None:
        raise ValueError(
            "intent target report not found "
            "in sender buffer"
        )

    if report_is_expired(
        report,
        current_step,
    ):
        raise ValueError(
            "cannot create transfer state "
            "for an expired report"
        )

    key = peer_transfer_key(
        intent.sender,
        intent.recipient,
        intent.target_id,
    )

    state = peer_transfer_states.get(
        key
    )

    if state is not None:
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        # A stale partial copy must not block a fresh report with the
        # same target_id after the original report lifetime expires.
        if peer_transfer_is_expired(
            state,
            current_step,
        ):
            del peer_transfer_states[key]
            state = None

    if state is not None:
        if (
            state.source_uav
            != report.source_uav
            or state.created_step
            != report.created_step
            or state.size_bytes
            != report.size_bytes
            or not np.isclose(
                state.ttl_s,
                report.ttl_s,
            )
        ):
            raise ValueError(
                "existing peer transfer state "
                "does not match sender report"
            )

        return state

    state = PeerTransferState(
        sender=int(intent.sender),
        recipient=int(intent.recipient),
        target_id=int(intent.target_id),
        source_uav=int(report.source_uav),
        created_step=int(report.created_step),
        size_bytes=int(report.size_bytes),
        ttl_s=float(report.ttl_s),
        received_bytes=0,
    )

    peer_transfer_states[key] = state

    return state

In [57]:
def commit_peer_transfer_bytes(
    intent,
    tx_bytes,
    sender_buffer,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if (
        isinstance(tx_bytes, (bool, np.bool_))
        or not isinstance(
            tx_bytes,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "tx_bytes must be an integer"
        )

    tx_bytes = int(tx_bytes)

    if tx_bytes < 0:
        raise ValueError(
            "tx_bytes must be >= 0"
        )

    if tx_bytes > intent.requested_bytes:
        raise ValueError(
            "tx_bytes cannot exceed "
            "intent.requested_bytes"
        )

    state = get_or_create_peer_transfer_state(
        intent,
        sender_buffer,
        peer_transfer_states,
        current_step,
    )

    if peer_transfer_is_expired(
        state,
        current_step,
    ):
        raise ValueError(
            "cannot commit bytes to an "
            "expired peer transfer"
        )

    remaining = (
        peer_transfer_remaining_bytes(
            state
        )
    )

    committed_bytes = min(
        tx_bytes,
        remaining,
    )

    state.received_bytes += (
        committed_bytes
    )

    return {
        "state": state,
        "committed_bytes": (
            committed_bytes
        ),
        "remaining_bytes": (
            peer_transfer_remaining_bytes(
                state
            )
        ),
        "complete": (
            peer_transfer_is_complete(
                state
            )
        ),
    }

In [ ]:
def cleanup_peer_transfer_states(
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    expired_keys = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if peer_transfer_is_expired(
            state,
            current_step,
        ):
            expired_keys.append(key)
            del peer_transfer_states[key]

    return expired_keys




def known_gcs_delivered_bytes_for_transfer(
    state,
    report_buffers,
):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    known_bytes = 0

    for buffer in report_buffers:
        if not isinstance(
            buffer,
            list,
        ):
            raise TypeError(
                "each report buffer must be a list"
            )

        for report in buffer:
            if not isinstance(
                report,
                Report,
            ):
                raise TypeError(
                    "report buffers must contain "
                    "only Report objects"
                )

            if report.target_id != state.target_id:
                continue

            # Only merge progress from the same report generation.
            if (
                report.source_uav
                != state.source_uav
                or report.created_step
                != state.created_step
                or report.size_bytes
                != state.size_bytes
                or not np.isclose(
                    report.ttl_s,
                    state.ttl_s,
                )
            ):
                continue

            known_bytes = max(
                known_bytes,
                int(
                    report.delivered_bytes
                ),
            )

    return min(
        known_bytes,
        int(state.size_bytes),
    )

def commit_completed_peer_transfer_to_receiver(
    state_key,
    peer_transfer_states,
    report_buffers,
    current_step,
    gcs_received_target_ids=None,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if len(report_buffers) != num_uavs:
        raise ValueError(
            "report_buffers length does not match "
            "CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(buffer, list):
            raise TypeError(
                "each report buffer must be a list"
            )

    if gcs_received_target_ids is None:
        gcs_received_target_ids = set()

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids must be a set"
        )

    if (
        not isinstance(state_key, tuple)
        or len(state_key) != 3
    ):
        raise TypeError(
            "state_key must be a "
            "(sender, recipient, target_id) tuple"
        )

    key = peer_transfer_key(
        state_key[0],
        state_key[1],
        state_key[2],
    )

    state = peer_transfer_states.get(
        key
    )

    if state is None:
        raise ValueError(
            "peer transfer state not found"
        )

    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "peer_transfer_states values must "
            "be PeerTransferState"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if not peer_transfer_is_complete(
        state
    ):
        return {
            "status": "incomplete",
            "report": None,
            "remaining_bytes": (
                peer_transfer_remaining_bytes(
                    state
                )
            ),
        }

    if peer_transfer_is_expired(
        state,
        current_step,
    ):
        del peer_transfer_states[key]

        return {
            "status": "expired",
            "report": None,
            "remaining_bytes": 0,
        }

    if (
        state.target_id
        in gcs_received_target_ids
    ):
        del peer_transfer_states[key]

        return {
            "status": "already_delivered",
            "report": None,
            "remaining_bytes": 0,
        }

    receiver_buffer = report_buffers[
        state.recipient
    ]

    cleanup_report_buffer(
        receiver_buffer,
        current_step,
    )

    known_gcs_delivered_bytes = (
        known_gcs_delivered_bytes_for_transfer(
            state,
            report_buffers,
        )
    )

    existing_report = find_report_in_buffer(
        receiver_buffer,
        state.target_id,
    )

    if existing_report is not None:
        existing_report.delivered_bytes = max(
            int(existing_report.delivered_bytes),
            int(known_gcs_delivered_bytes),
        )
        # The receiver already has a full copy. Keep the completed
        # transfer state so this sender does not retransmit it.
        return {
            "status": "already_present",
            "report": existing_report,
            "remaining_bytes": 0,
        }

    received_report = Report(
        target_id=state.target_id,
        source_uav=state.source_uav,
        created_step=state.created_step,
        size_bytes=state.size_bytes,
        ttl_s=state.ttl_s,
        delivered_bytes=(
            known_gcs_delivered_bytes
        ),
    )

    enqueued, reason = enqueue_report(
        receiver_buffer,
        received_report,
    )

    if enqueued:
        # Keep the completed state as a lightweight receipt that this
        # sender already transferred the full report to this recipient.
        # build_transmission_intent() will therefore return None for the
        # same hop instead of sending the same report again.
        return {
            "status": "enqueued",
            "report": received_report,
            "remaining_bytes": 0,
        }

    if reason == "buffer_full":
        return {
            "status": "buffer_full",
            "report": None,
            "remaining_bytes": 0,
        }

    if reason == "duplicate":
        existing_report = find_report_in_buffer(
            receiver_buffer,
            state.target_id,
        )

        return {
            "status": "already_present",
            "report": existing_report,
            "remaining_bytes": 0,
        }

    raise RuntimeError(
        f"unexpected enqueue result: {reason}"
    )

In [ ]:
def reports_are_same_generation(
    report_a,
    report_b,
):
    if not isinstance(
        report_a,
        Report,
    ):
        raise TypeError(
            "report_a must be Report"
        )

    if not isinstance(
        report_b,
        Report,
    ):
        raise TypeError(
            "report_b must be Report"
        )

    return bool(
        report_a.target_id
        == report_b.target_id
        and report_a.source_uav
        == report_b.source_uav
        and report_a.created_step
        == report_b.created_step
        and report_a.size_bytes
        == report_b.size_bytes
        and np.isclose(
            report_a.ttl_s,
            report_b.ttl_s,
        )
    )


def remove_peer_transfer_states_for_target(
    peer_transfer_states,
    target_id,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(target_id, (bool, np.bool_))
        or not isinstance(
            target_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "target_id must be an integer"
        )

    target_id = int(target_id)

    if target_id < 0:
        raise ValueError(
            "target_id must be >= 0"
        )

    removed_keys = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if state.target_id == target_id:
            removed_keys.append(key)
            del peer_transfer_states[key]

    return removed_keys


def commit_gcs_transfer_bytes(
    intent,
    tx_bytes,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if intent.recipient != GCS_NODE:
        raise ValueError(
            "commit_gcs_transfer_bytes "
            "requires a GCS intent"
        )

    if (
        isinstance(tx_bytes, (bool, np.bool_))
        or not isinstance(
            tx_bytes,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "tx_bytes must be an integer"
        )

    tx_bytes = int(tx_bytes)

    if tx_bytes < 0:
        raise ValueError(
            "tx_bytes must be >= 0"
        )

    if tx_bytes > intent.requested_bytes:
        raise ValueError(
            "tx_bytes cannot exceed "
            "intent.requested_bytes"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(
            buffer,
            list,
        ):
            raise TypeError(
                "each report buffer "
                "must be a list"
            )

        for report in buffer:
            if not isinstance(
                report,
                Report,
            ):
                raise TypeError(
                    "report buffers must contain "
                    "only Report objects"
                )

    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    for report in pending_reports:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "pending_reports must contain "
                "only Report objects"
            )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(
        current_step
    )

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    if (
        intent.target_id
        in gcs_received_target_ids
    ):
        # Make this path idempotent: if stale network copies remain for
        # any reason, calling the commit again restores the invariant
        # that a GCS-delivered target has no buffered/pending copies.
        cleanup_result = (
            mark_target_delivered_to_gcs(
                intent.target_id,
                gcs_received_target_ids,
                report_buffers,
                pending_reports,
            )
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

        return {
            "status": "already_delivered",
            "committed_bytes": 0,
            "delivered_bytes": 0,
            "remaining_bytes": 0,
            "removed_peer_states": (
                removed_peer_states
            ),
            "cleanup_result": (
                cleanup_result
            ),
        }

    sender_buffer = report_buffers[
        intent.sender
    ]

    sender_report = (
        find_report_in_buffer(
            sender_buffer,
            intent.target_id,
        )
    )

    if sender_report is None:
        raise ValueError(
            "intent target report "
            "not found in sender buffer"
        )

    if report_is_expired(
        sender_report,
        current_step,
    ):
        raise ValueError(
            "cannot commit an "
            "expired report to GCS"
        )

    report_copies = []

    for buffer in report_buffers:
        for report in buffer:
            if reports_are_same_generation(
                report,
                sender_report,
            ):
                report_copies.append(
                    report
                )

    if not report_copies:
        raise RuntimeError(
            "no matching report copies found"
        )

    known_delivered_bytes = max(
        int(report.delivered_bytes)
        for report in report_copies
    )

    remaining_bytes = max(
        0,
        int(sender_report.size_bytes)
        - known_delivered_bytes,
    )

    committed_bytes = min(
        tx_bytes,
        remaining_bytes,
    )

    new_delivered_bytes = (
        known_delivered_bytes
        + committed_bytes
    )

    for report in report_copies:
        report.delivered_bytes = (
            new_delivered_bytes
        )

    complete = (
        new_delivered_bytes
        >= sender_report.size_bytes
    )

    removed_peer_states = []

    if complete:
        mark_target_delivered_to_gcs(
            intent.target_id,
            gcs_received_target_ids,
            report_buffers,
            pending_reports,
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

    return {
        "status": (
            "delivered"
            if complete
            else "partial"
        ),
        "committed_bytes": (
            committed_bytes
        ),
        "delivered_bytes": (
            new_delivered_bytes
        ),
        "remaining_bytes": max(
            0,
            int(sender_report.size_bytes)
            - new_delivered_bytes,
        ),
        "removed_peer_states": (
            removed_peer_states
        ),
    }


In [ ]:
def retry_completed_peer_transfers(
    peer_transfer_states,
    report_buffers,
    current_step,
    gcs_received_target_ids,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(buffer, list):
            raise TypeError(
                "each report buffer must be a list"
            )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    results = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if not peer_transfer_is_complete(
            state
        ):
            continue

        receiver_buffer = report_buffers[
            state.recipient
        ]

        # Remove stale receiver copies before deciding whether the
        # completed state can act as a receipt.
        cleanup_report_buffer(
            receiver_buffer,
            current_step,
        )

        # Expiration and global GCS delivery take precedence over receipt
        # retention. commit_completed_peer_transfer_to_receiver() removes
        # the completed state in both cases.
        if (
            peer_transfer_is_expired(
                state,
                current_step,
            )
            or state.target_id
            in gcs_received_target_ids
        ):
            result = (
                commit_completed_peer_transfer_to_receiver(
                    key,
                    peer_transfer_states,
                    report_buffers,
                    current_step,
                    gcs_received_target_ids,
                )
            )

            results.append(
                {
                    "key": key,
                    "status": result["status"],
                    "result": result,
                }
            )
            continue

        # A non-expired completed state is also used as a receipt after
        # the receiver has a full copy. Do not repeatedly re-commit it.
        existing_report = find_report_in_buffer(
            receiver_buffer,
            state.target_id,
        )

        if (
            existing_report is not None
            and reports_are_same_generation(
                existing_report,
                Report(
                    target_id=state.target_id,
                    source_uav=state.source_uav,
                    created_step=state.created_step,
                    size_bytes=state.size_bytes,
                    ttl_s=state.ttl_s,
                    delivered_bytes=0,
                ),
            )
        ):
            results.append(
                {
                    "key": key,
                    "status": "receipt_present",
                }
            )
            continue

        result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                peer_transfer_states,
                report_buffers,
                current_step,
                gcs_received_target_ids,
            )
        )

        results.append(
            {
                "key": key,
                "status": result["status"],
                "result": result,
            }
        )

    return results


def execute_transmission_intent(
    intent,
    uavs,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
    peer_transfer_states,
    current_step,
    obstacles=None,
    dt=None,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    if obstacles is None:
        obstacles = []

    if not isinstance(obstacles, list):
        raise TypeError(
            "obstacles must be a list"
        )

    retry_results = (
        retry_completed_peer_transfers(
            peer_transfer_states,
            report_buffers,
            current_step,
            gcs_received_target_ids,
        )
    )

    # A stale intent can survive after another copy reaches GCS.
    # Restore the global-delivery invariant before doing any radio work.
    if (
        intent.target_id
        in gcs_received_target_ids
    ):
        cleanup_result = (
            mark_target_delivered_to_gcs(
                intent.target_id,
                gcs_received_target_ids,
                report_buffers,
                pending_reports,
            )
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

        return {
            "status": "already_delivered",
            "tx_bytes": 0,
            "retry_results": retry_results,
            "cleanup_result": cleanup_result,
            "removed_peer_states": (
                removed_peer_states
            ),
        }

    tx_bytes = calculate_intent_tx_bytes(
        intent,
        uavs,
        obstacles=obstacles,
        dt=dt,
    )

    if tx_bytes <= 0:
        return {
            "status": "no_transfer",
            "tx_bytes": 0,
            "retry_results": retry_results,
        }

    if intent.recipient == GCS_NODE:
        commit_result = (
            commit_gcs_transfer_bytes(
                intent,
                tx_bytes,
                report_buffers,
                pending_reports,
                gcs_received_target_ids,
                peer_transfer_states,
                current_step,
            )
        )

        return {
            "status": (
                "gcs_"
                + commit_result["status"]
            ),
            "tx_bytes": tx_bytes,
            "retry_results": retry_results,
            "commit_result": commit_result,
        }

    sender_buffer = report_buffers[
        intent.sender
    ]

    peer_result = (
        commit_peer_transfer_bytes(
            intent,
            tx_bytes,
            sender_buffer,
            peer_transfer_states,
            current_step,
        )
    )

    custody_result = None

    if peer_result["complete"]:
        key = peer_transfer_key(
            intent.sender,
            intent.recipient,
            intent.target_id,
        )

        custody_result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                peer_transfer_states,
                report_buffers,
                current_step,
                gcs_received_target_ids,
            )
        )

    if custody_result is None:
        status = "peer_partial"
    else:
        status = (
            "peer_"
            + custody_result["status"]
        )

    return {
        "status": status,
        "tx_bytes": tx_bytes,
        "retry_results": retry_results,
        "peer_result": peer_result,
        "custody_result": custody_result,
    }
